In [1]:
import pandas as pd

In [2]:
excel_path = r"D:\ai_powered_data_analyst\Retail_Analytics\data\online_retail_II.xlsx"

In [3]:
# List every worksheet and its dimensions
xls = pd.ExcelFile(excel_path)

print("Worksheet names:", xls.sheet_names)

for sheet in xls.sheet_names:
    temp = pd.read_excel(excel_path, sheet_name=sheet)

    print(f"\nWorksheet: {sheet}")
    print("Rows:", len(temp))
    print("Columns:", len(temp.columns))

    if "InvoiceDate" in temp.columns:
        dates = pd.to_datetime(temp["InvoiceDate"], errors="coerce")
        print("Date range:", dates.min(), "to", dates.max())


Worksheet names: ['Year 2009-2010', 'Year 2010-2011']

Worksheet: Year 2009-2010
Rows: 525461
Columns: 8
Date range: 2009-12-01 07:45:00 to 2010-12-09 20:01:00

Worksheet: Year 2010-2011
Rows: 541910
Columns: 8
Date range: 2010-12-01 08:26:00 to 2011-12-09 12:50:00


In [4]:
from pathlib import Path

output_path = Path(r"D:\retail-analytics\data\raw\online_retail_full.csv")
# Load both worksheets explicitly
year_1 = pd.read_excel(excel_path, sheet_name="Year 2009-2010")
year_2 = pd.read_excel(excel_path, sheet_name="Year 2010-2011")

print("Year 2009-2010:", year_1.shape)
print("Year 2010-2011:", year_2.shape)

# Check that both worksheets have the same columns
if list(year_1.columns) != list(year_2.columns):
    raise ValueError("Worksheet columns do not match. Inspect before combining.")

# Combine every row; do not remove duplicates at this stage
full_df = pd.concat([year_1, year_2], ignore_index=True)

# Standardise the date column's data type
full_df["InvoiceDate"] = pd.to_datetime(
    full_df["InvoiceDate"], errors="coerce"
)

# Save a separate combined file, preserving the original workbook and CSV
output_path.parent.mkdir(parents=True, exist_ok=True)
full_df.to_csv(output_path, index=False)

print("\n--- COMBINED DATASET VALIDATION ---")
print("Total rows:", len(full_df))
print("Total columns:", len(full_df.columns))
print("Date range:", full_df["InvoiceDate"].min(),
      "to", full_df["InvoiceDate"].max())
print("Exact duplicate rows:", full_df.duplicated().sum())
print("Missing customer IDs:", full_df["Customer ID"].isna().sum())
print("Saved to:", output_path)

Year 2009-2010: (525461, 8)
Year 2010-2011: (541910, 8)

--- COMBINED DATASET VALIDATION ---
Total rows: 1067371
Total columns: 8
Date range: 2009-12-01 07:45:00 to 2011-12-09 12:50:00
Exact duplicate rows: 34335
Missing customer IDs: 243007
Saved to: D:\retail-analytics\data\raw\online_retail_full.csv


In [1]:

import pandas as pd
from pathlib import Path

# Load the combined two-year dataset
data_path = Path(r"D:\ai_powered_data_analyst\Retail_Analytics\data\online_retail_full.csv")

df_full = pd.read_csv(data_path, low_memory=False)
df_full["InvoiceDate"] = pd.to_datetime(
    df_full["InvoiceDate"], errors="coerce"
)

# 1. Basic structure
print("=== DATASET OVERVIEW ===")
print("Rows:", len(df_full))
print("Columns:", len(df_full.columns))
print("Column names:", df_full.columns.tolist())
print("Date range:",
      df_full["InvoiceDate"].min(), "to",
      df_full["InvoiceDate"].max())
print("Unique invoices:", df_full["Invoice"].nunique())
print("Unique customers:", df_full["Customer ID"].nunique())

# 2. Missing values
print("\n=== MISSING VALUES ===")
missing_report = pd.DataFrame({
    "Missing count": df_full.isna().sum(),
    "Missing percentage": (
        df_full.isna().mean() * 100
    ).round(2)
})
display(missing_report)

# 3. Duplicate records
print("\n=== DUPLICATE CHECK ===")
print("Exact duplicate rows:", df_full.duplicated().sum())
print("Rows with missing invoice dates:",
      df_full["InvoiceDate"].isna().sum())

# 4. Transaction and price diagnostics
invoice_text = df_full["Invoice"].astype(str).str.strip()

checks = {
    "C-prefixed invoice": invoice_text.str.startswith("C"),
    "Negative quantity": df_full["Quantity"] < 0,
    "Zero quantity": df_full["Quantity"] == 0,
    "Zero price": df_full["Price"] == 0,
    "Negative price": df_full["Price"] < 0,
    "Missing customer ID": df_full["Customer ID"].isna(),
    "Missing description": df_full["Description"].isna()
}

quality_rows = []

for label, mask in checks.items():
    quality_rows.append({
        "Check": label,
        "Record count": int(mask.sum()),
        "Percentage": round(mask.mean() * 100, 2)
    })

display(pd.DataFrame(quality_rows))

# 5. Numeric summary
print("\n=== QUANTITY AND PRICE SUMMARY ===")
display(df_full[["Quantity", "Price"]].describe().round(2))

# 6. Save a quality report, not a cleaned dataset
report_path = Path(
    r"D:\ai_powered_data_analyst\Retail_Analytics\reports\data_quality_summary.csv"
)
report_path.parent.mkdir(parents=True, exist_ok=True)

pd.DataFrame(quality_rows).to_csv(report_path, index=False)

print("\nQuality summary saved to:", report_path)


=== DATASET OVERVIEW ===
Rows: 1067371
Columns: 8
Column names: ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']
Date range: 2009-12-01 07:45:00 to 2011-12-09 12:50:00
Unique invoices: 53628
Unique customers: 5942

=== MISSING VALUES ===


,Missing count,Missing percentage
Invoice,0,0.00
StockCode,0,0.00
Description,4382,0.41
Quantity,0,0.00
InvoiceDate,0,0.00
Price,0,0.00
Customer ID,243007,22.77
Country,0,0.00



=== DUPLICATE CHECK ===
Exact duplicate rows: 34335
Rows with missing invoice dates: 0


,Check,Record count,Percentage
0,C-prefixed invoice,19494,1.83
1,Negative quantity,22950,2.15
2,Zero quantity,0,0.00
3,Zero price,6202,0.58
4,Negative price,5,0.00
5,Missing customer ID,243007,22.77
6,Missing description,4382,0.41



=== QUANTITY AND PRICE SUMMARY ===


,Quantity,Price
count,1067371.00,1067371.00
mean,9.94,4.65
std,172.71,123.55
min,-80995.00,-53594.36
25%,1.00,1.25
50%,3.00,2.10
75%,10.00,4.15
max,80995.00,38970.00



Quality summary saved to: D:\ai_powered_data_analyst\Retail_Analytics\reports\data_quality_summary.csv


In [3]:

import pandas as pd

# Work on a diagnostic copy
audit = df_full.copy()

audit["Invoice"] = audit["Invoice"].astype(str).str.strip()
audit["LineValue"] = audit["Quantity"] * audit["Price"]
audit["IsCancellationInvoice"] = audit["Invoice"].str.startswith("C")

# 1. Understand the duplicate counts
print("=== DUPLICATE INVESTIGATION ===")

duplicate_mask = audit.duplicated(keep=False)

print("Rows beyond the first occurrence:", audit.duplicated().sum())
print("Rows belonging to duplicate groups:", duplicate_mask.sum())
print(
    "Distinct duplicate groups:",
    audit.loc[duplicate_mask].drop_duplicates().shape[0]
)

print("\nExamples of exact duplicate records:")
display(
    audit.loc[duplicate_mask]
    .sort_values(["Invoice", "StockCode", "InvoiceDate"])
    .head(15)
)

# 2. Inspect all negative-price rows
print("\n=== ALL NEGATIVE-PRICE RECORDS ===")

display(
    audit.loc[audit["Price"] < 0, [
        "Invoice", "StockCode", "Description",
        "Quantity", "InvoiceDate", "Price",
        "Customer ID", "Country", "LineValue"
    ]].sort_values("Price")
)

# 3. Inspect the largest absolute quantities
print("\n=== 15 LARGEST ABSOLUTE QUANTITIES ===")

display(
    audit.loc[audit["Quantity"].abs().nlargest(15).index, [
        "Invoice", "StockCode", "Description",
        "Quantity", "InvoiceDate", "Price",
        "Customer ID", "Country", "LineValue"
    ]].sort_values("Quantity", key=lambda s: s.abs(), ascending=False)
)

# 4. Inspect the largest absolute prices
print("\n=== 15 LARGEST ABSOLUTE PRICES ===")

display(
    audit.loc[audit["Price"].abs().nlargest(15).index, [
        "Invoice", "StockCode", "Description",
        "Quantity", "InvoiceDate", "Price",
        "Customer ID", "Country", "LineValue"
    ]].sort_values("Price", key=lambda s: s.abs(), ascending=False)
)

# 5. Check whether C-prefixed invoices consistently have negative quantities
print("\n=== CANCELLATION-PREFIX CROSS-CHECK ===")

audit["QuantityType"] = audit["Quantity"].map(
    lambda q: "Negative" if q < 0 else "Positive"
)

display(
    pd.crosstab(
        audit["IsCancellationInvoice"],
        audit["QuantityType"],
        margins=True
    )
)

# 6. Compare recorded line values by invoice prefix
print("\n=== RECORDED LINE VALUE BY INVOICE PREFIX ===")

display(
    audit.groupby("IsCancellationInvoice").agg(
        RecordCount=("Invoice", "size"),
        InvoiceCount=("Invoice", "nunique"),
        TotalQuantity=("Quantity", "sum"),
        RecordedLineValue=("LineValue", "sum"),
        MissingCustomerRows=("Customer ID", lambda s: s.isna().sum())
    ).round(2)
)


=== DUPLICATE INVESTIGATION ===
Rows beyond the first occurrence: 34335
Rows belonging to duplicate groups: 67242
Distinct duplicate groups: 32907

Examples of exact duplicate records:


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,LineValue,IsCancellationInvoice
379,489517,21491,SET OF THREE VINTAGE GIFT WRAPS,1,2009-12-01 11:34:00,1.95,16329.0,United Kingdom,1.95,False
391,489517,21491,SET OF THREE VINTAGE GIFT WRAPS,1,2009-12-01 11:34:00,1.95,16329.0,United Kingdom,1.95,False
365,489517,21821,GLITTER STAR GARLAND WITH BELLS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom,3.75,False
386,489517,21821,GLITTER STAR GARLAND WITH BELLS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom,3.75,False
363,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom,3.75,False
371,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom,3.75,False
394,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom,3.75,False
362,489517,21913,VINTAGE SEASIDE JIGSAW PUZZLES,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom,3.75,False
385,489517,21913,VINTAGE SEASIDE JIGSAW PUZZLES,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom,3.75,False
368,489517,22130,PARTY CONE CHRISTMAS DECORATION,6,2009-12-01 11:34:00,0.85,16329.0,United Kingdom,5.10,False



=== ALL NEGATIVE-PRICE RECORDS ===


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,LineValue
179403,A506401,B,Adjust bad debt,1,2010-04-29 13:36:00,-53594.36,NaN,United Kingdom,-53594.36
276274,A516228,B,Adjust bad debt,1,2010-07-19 11:24:00,-44031.79,NaN,United Kingdom,-44031.79
403472,A528059,B,Adjust bad debt,1,2010-10-20 12:04:00,-38925.87,NaN,United Kingdom,-38925.87
825444,A563186,B,Adjust bad debt,1,2011-08-12 14:51:00,-11062.06,NaN,United Kingdom,-11062.06
825445,A563187,B,Adjust bad debt,1,2011-08-12 14:52:00,-11062.06,NaN,United Kingdom,-11062.06



=== 15 LARGEST ABSOLUTE QUANTITIES ===


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,LineValue
1065882,581483,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,2011-12-09 09:15:00,2.08,16446.0,United Kingdom,168469.6
1065883,C581484,23843,"PAPER CRAFT , LITTLE BIRDIE",-80995,2011-12-09 09:27:00,2.08,16446.0,United Kingdom,-168469.6
587080,541431,23166,MEDIUM CERAMIC TOP STORAGE JAR,74215,2011-01-18 10:01:00,1.04,12346.0,United Kingdom,77183.6
587085,C541433,23166,MEDIUM CERAMIC TOP STORAGE JAR,-74215,2011-01-18 10:17:00,1.04,12346.0,United Kingdom,-77183.6
90857,497946,37410,BLACK AND WHITE PAISLEY FLOWER MUG,19152,2010-02-15 11:57:00,0.10,13902.0,Denmark,1915.2
127166,501534,21099,SET/6 STRAWBERRY PAPER CUPS,12960,2010-03-17 13:09:00,0.10,13902.0,Denmark,1296.0
127168,501534,21091,SET/6 WOODLAND PAPER PLATES,12960,2010-03-17 13:09:00,0.10,13902.0,Denmark,1296.0
127169,501534,21085,SET/6 WOODLAND PAPER CUPS,12744,2010-03-17 13:09:00,0.10,13902.0,Denmark,1274.4
1027583,578841,84826,ASSTD DESIGN 3D PAPER STICKERS,12540,2011-11-25 15:57:00,0.00,13256.0,United Kingdom,0.0
127167,501534,21092,SET/6 STRAWBERRY PAPER PLATES,12480,2010-03-17 13:09:00,0.10,13902.0,Denmark,1248.0


... (2 outputs left out)


=== CANCELLATION-PREFIX CROSS-CHECK ===


QuantityType,Negative,Positive,All
IsCancellationInvoice,,,
False,3457,1044420,1047877
True,19493,1,19494
All,22950,1044421,1067371



=== RECORDED LINE VALUE BY INVOICE PREFIX ===


,RecordCount,InvoiceCount,TotalQuantity,RecordedLineValue,MissingCustomerRows
IsCancellationInvoice,,,,,
False,1047877,45336,11099484,20813918.43,242257
True,19494,8292,-490992,-1526667.86,750


In [4]:

# Create review flags without changing or deleting any records
audit = df_full.copy()

invoice_text = audit["Invoice"].astype(str).str.strip()
description = audit["Description"].fillna("").str.strip().str.lower()
stock_code = audit["StockCode"].astype(str).str.strip().str.upper()

audit["LineValue"] = audit["Quantity"] * audit["Price"]

audit["C_invoice"] = invoice_text.str.startswith("C")
audit["Negative_quantity"] = audit["Quantity"] < 0
audit["Negative_price"] = audit["Price"] < 0
audit["Zero_price"] = audit["Price"] == 0
audit["Bad_debt_adjustment"] = description.eq("adjust bad debt")
audit["Manual_entry"] = description.eq("manual")
audit["Bank_charge"] = description.str.contains("bank charge", regex=False)
audit["Amazon_fee"] = description.str.contains("amazon fee", regex=False)

# Summarise categories separately; categories may overlap
flag_columns = [
    "C_invoice",
    "Negative_quantity",
    "Negative_price",
    "Zero_price",
    "Bad_debt_adjustment",
    "Manual_entry",
    "Bank_charge",
    "Amazon_fee"
]

review_summary = pd.DataFrame([
    {
        "Record type": col,
        "Rows": int(audit[col].sum()),
        "Recorded line value": round(audit.loc[audit[col], "LineValue"].sum(), 2)
    }
    for col in flag_columns
])

display(review_summary)

# Inspect the duplicate-heavy example invoice
print("\n=== INVOICE 489517 ===")
display(
    audit.loc[invoice_text.eq("489517")].sort_values(
        ["StockCode", "Description"]
    )
)

# Save the classification report
from pathlib import Path

report_path = Path(
    r"D:\ai_powered_data_analyst\Retail_Analytics\reports\record_classification_audit.csv"
)
report_path.parent.mkdir(parents=True, exist_ok=True)
review_summary.to_csv(report_path, index=False)

print("\nSaved to:", report_path)


,Record type,Rows,Recorded line value
0,C_invoice,19494,-1526667.86
1,Negative_quantity,22950,-1527041.43
2,Negative_price,5,-158676.14
3,Zero_price,6202,0.00
4,Bad_debt_adjustment,6,-147614.08
5,Manual_entry,1426,-82781.27
6,Bank_charge,102,-35562.63
7,Amazon_fee,43,-260763.58



=== INVOICE 489517 ===


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,LineValue,C_invoice,Negative_quantity,Negative_price,Zero_price,Bad_debt_adjustment,Manual_entry,Bank_charge,Amazon_fee
364,489517,16207A,PINK STRAWBERRY HANDBAG,2,2009-12-01 11:34:00,2.95,16329.0,United Kingdom,5.90,False,False,False,False,False,False,False,False
398,489517,20675,BLUE SPOTTY BOWL,4,2009-12-01 11:34:00,1.25,16329.0,United Kingdom,5.00,False,False,False,False,False,False,False,False
375,489517,20711,JUMBO BAG TOYS,1,2009-12-01 11:34:00,1.95,16329.0,United Kingdom,1.95,False,False,False,False,False,False,False,False
381,489517,20971,PINK BLUE FELT CRAFT TRINKET BOX,1,2009-12-01 11:34:00,1.25,16329.0,United Kingdom,1.25,False,False,False,False,False,False,False,False
370,489517,20972,PINK CREAM FELT CRAFT TRINKET BOX,1,2009-12-01 11:34:00,1.25,16329.0,United Kingdom,1.25,False,False,False,False,False,False,False,False
397,489517,21244,BLUE SPOTTY PLATE,4,2009-12-01 11:34:00,1.69,16329.0,United Kingdom,6.76,False,False,False,False,False,False,False,False
382,489517,21252,SET OF MEADOW FLOWER STICKERS,1,2009-12-01 11:34:00,2.95,16329.0,United Kingdom,2.95,False,False,False,False,False,False,False,False
379,489517,21491,SET OF THREE VINTAGE GIFT WRAPS,1,2009-12-01 11:34:00,1.95,16329.0,United Kingdom,1.95,False,False,False,False,False,False,False,False
391,489517,21491,SET OF THREE VINTAGE GIFT WRAPS,1,2009-12-01 11:34:00,1.95,16329.0,United Kingdom,1.95,False,False,False,False,False,False,False,False
369,489517,21584,RETRO SPOT SMALL TUBE MATCHES,20,2009-12-01 11:34:00,1.65,16329.0,United Kingdom,33.00,False,False,False,False,False,False,False,False



Saved to: D:\ai_powered_data_analyst\Retail_Analytics\reports\record_classification_audit.csv


In [5]:
audit = df_full.copy()

audit["Invoice"] = audit["Invoice"].astype(str).str.strip()

audit["Description_clean"] = (
    audit["Description"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)

audit["LineValue"] = audit["Quantity"] * audit["Price"]

print("Columns created successfully:")
print(audit.columns.tolist())

print("\nData shape:", audit.shape)

Columns created successfully:
['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country', 'Description_clean', 'LineValue']

Data shape: (1067371, 10)


In [6]:
# Count exact duplicate rows
duplicate_count = audit.duplicated().sum()

# Preview the dataset with exact duplicates removed.
# This is only a temporary assessment; we are NOT changing audit.
deduplicated_preview = audit.drop_duplicates()

print("Original row count:", len(audit))
print("Exact duplicate rows:", duplicate_count)
print("Rows after removing exact duplicates:", len(deduplicated_preview))
print("Rows potentially removed:", len(audit) - len(deduplicated_preview))

# Compare recorded line values
print("\nOriginal total line value:", round(audit["LineValue"].sum(), 2))
print(
    "Line value after duplicate removal:",
    round(deduplicated_preview["LineValue"].sum(), 2)
)
print(
    "Difference in line value:",
    round(
        audit["LineValue"].sum() - deduplicated_preview["LineValue"].sum(),
        2
    )
)

Original row count: 1067371
Exact duplicate rows: 34335
Rows after removing exact duplicates: 1033036
Rows potentially removed: 34335

Original total line value: 19287250.57
Line value after duplicate removal: 18855533.7
Difference in line value: 431716.87


In [7]:
special_mask = audit["Description_clean"].str.contains(
    r"adjust bad debt|manual|bank charge|amazon fee",
    regex=True,
    na=False
)

special_records = audit.loc[
    special_mask,
    [
        "Invoice",
        "StockCode",
        "Description",
        "Quantity",
        "InvoiceDate",
        "Price",
        "Customer ID",
        "Country",
        "LineValue"
    ]
]

print("Special records found:", len(special_records))

display(special_records.head(30))

Special records found: 1577


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,LineValue
2697,489609,M,Manual,1,2009-12-01 14:50:00,4.00,NaN,United Kingdom,4.00
3053,C489651,M,Manual,-1,2009-12-01 16:48:00,5.10,17804.0,United Kingdom,-5.10
5897,C489859,M,Manual,-1,2009-12-02 14:45:00,69.57,NaN,United Kingdom,-69.57
9259,C490126,M,Manual,-1,2009-12-03 18:12:00,5.95,15884.0,United Kingdom,-5.95
9307,C490129,M,Manual,-1,2009-12-03 18:26:00,1998.49,15482.0,United Kingdom,-1998.49
11310,490300,M,Manual,1,2009-12-04 14:19:00,0.85,12970.0,United Kingdom,0.85
11311,490300,M,Manual,1,2009-12-04 14:19:00,0.21,12970.0,United Kingdom,0.21
16107,490727,M,Manual,1,2009-12-07 16:38:00,0.00,17231.0,United Kingdom,0.00
17273,C490748,M,Manual,-1,2009-12-07 18:14:00,309.73,12748.0,United Kingdom,-309.73
17386,490760,M,Manual,1,2009-12-08 09:49:00,10.00,14295.0,United Kingdom,10.00


In [8]:
audit["IsCancellationInvoice"] = (
    audit["Invoice"].str.startswith("C", na=False)
)

customer_id_summary = (
    audit.groupby("IsCancellationInvoice")
    .agg(
        total_rows=("Invoice", "size"),
        missing_customer_ids=("Customer ID", lambda x: x.isna().sum()),
        unique_invoices=("Invoice", "nunique")
    )
    .reset_index()
)

customer_id_summary["missing_customer_pct"] = (
    customer_id_summary["missing_customer_ids"]
    / customer_id_summary["total_rows"]
    * 100
).round(2)

display(customer_id_summary)


,IsCancellationInvoice,total_rows,missing_customer_ids,unique_invoices,missing_customer_pct
0,False,1047877,242257,45336,23.12
1,True,19494,750,8292,3.85


In [9]:
# Work only with a copy; do not modify audit or df_full.
duplicate_audit = audit.copy()

# Identify exact duplicate rows
duplicate_audit["IsExactDuplicate"] = audit.duplicated(keep="first")

# Identify cancellation invoice prefix
duplicate_audit["IsCancellationInvoice"] = (
    duplicate_audit["Invoice"].astype(str).str.startswith("C")
)

# Identify special descriptions
duplicate_audit["IsSpecialDescription"] = (
    duplicate_audit["Description_clean"].str.contains(
        r"adjust bad debt|manual|bank charge|amazon fee",
        regex=True,
        na=False
    )
)

# Summarize duplicate rows by category
duplicate_summary = (
    duplicate_audit
    .groupby(
        [
            "IsCancellationInvoice",
            "IsSpecialDescription",
            "IsExactDuplicate"
        ],
        dropna=False
    )
    .agg(
        row_count=("Invoice", "size"),
        line_value=("LineValue", "sum")
    )
    .reset_index()
)

duplicate_summary["line_value"] = (
    duplicate_summary["line_value"].round(2)
)

display(duplicate_summary)

,IsCancellationInvoice,IsSpecialDescription,IsExactDuplicate,row_count,line_value
0,False,False,False,1013031,20104970.06
1,False,False,True,33913,481287.75
2,False,True,False,901,212614.25
3,False,True,True,32,15046.37
4,True,False,False,18471,-761508.25
5,True,False,True,379,-10777.43
6,True,True,False,633,-700542.36
7,True,True,True,11,-53839.82


In [11]:
# Count how many times each exact original row occurs.
original_columns = [
    "Invoice",
    "StockCode",
    "Description",
    "Quantity",
    "InvoiceDate",
    "Price",
    "Customer ID",
    "Country"
]

duplicate_groups = (
    audit.groupby(original_columns, dropna=False)
    .size()
    .reset_index(name="OccurrenceCount")
)

duplicate_groups = (
    duplicate_groups[duplicate_groups["OccurrenceCount"] > 1]
    .sort_values("OccurrenceCount", ascending=False)
)

print("Distinct repeated row patterns:", len(duplicate_groups))
print(
    "Rows belonging to repeated groups:",
    duplicate_groups["OccurrenceCount"].sum()
)

display(duplicate_groups.head(20))

Distinct repeated row patterns: 32907
Rows belonging to repeated groups: 67242


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,OccurrenceCount
694118,555524,22698,PINK REGENCY TEACUP AND SAUCER,1,2011-06-05 11:37:00,2.95,16923.0,United Kingdom,20
694117,555524,22697,GREEN REGENCY TEACUP AND SAUCER,1,2011-06-05 11:37:00,2.95,16923.0,United Kingdom,12
496282,537224,70007,HI TEC ALPINE HAND WARMER,1,2010-12-05 16:24:00,1.65,13174.0,United Kingdom,10
895184,572861,22775,PURPLE DRAWERKNOB ACRYLIC EDWARDIAN,12,2011-10-26 12:46:00,1.25,14102.0,United Kingdom,8
503982,537781,22898,CHILDRENS APRON APPLES DESIGN,1,2010-12-08 12:46:00,1.95,17341.0,United Kingdom,6
497953,537265,79000,MOROCCAN TEA GLASS,12,2010-12-06 11:26:00,0.85,15919.0,United Kingdom,6
490565,536749,22174,PHOTO CUBE,2,2010-12-02 13:49:00,1.65,17976.0,United Kingdom,6
503969,537781,22745,POPPY'S PLAYHOUSE BEDROOM,1,2010-12-08 12:46:00,2.10,17341.0,United Kingdom,6
495877,537208,90082B,DIAMANTE BOW BROOCH RED COLOUR,1,2010-12-05 15:12:00,6.35,15889.0,United Kingdom,6
499324,537422,22585,PACK OF 6 BIRDY GIFT TAGS,2,2010-12-06 15:27:00,1.25,17812.0,United Kingdom,6


In [12]:
# Inspect the complete invoices containing highly repeated rows.

invoice_numbers = ["555524", "537781", "537144"]

invoice_review = audit.loc[
    audit["Invoice"].astype(str).isin(invoice_numbers),
    [
        "Invoice",
        "InvoiceDate",
        "Customer ID",
        "StockCode",
        "Description",
        "Quantity",
        "Price",
        "LineValue"
    ]
].sort_values(
    ["Invoice", "StockCode", "Description"]
)

display(invoice_review.to_string(index=False))

"Invoice         InvoiceDate  Customer ID StockCode                         Description  Quantity  Price  LineValue\n 537144 2010-12-05 13:00:00      15880.0    15060B          FAIRY CAKE DESIGN UMBRELLA         1   3.75       3.75\n 537144 2010-12-05 13:00:00      15880.0    15060B          FAIRY CAKE DESIGN UMBRELLA         1   3.75       3.75\n 537144 2010-12-05 13:00:00      15880.0     20764        ABSTRACT CIRCLES SKETCHBOOK          1   3.75       3.75\n 537144 2010-12-05 13:00:00      15880.0     20764        ABSTRACT CIRCLES SKETCHBOOK          1   3.75       3.75\n 537144 2010-12-05 13:00:00      15880.0     21110         LARGE CAKE TOWEL PINK SPOTS         1   6.75       6.75\n 537144 2010-12-05 13:00:00      15880.0     21110         LARGE CAKE TOWEL PINK SPOTS         1   6.75       6.75\n 537144 2010-12-05 13:00:00      15880.0     21158             MOODY GIRL DOOR HANGER          1   1.45       1.45\n 537144 2010-12-05 13:00:00      15880.0     21158             MOODY GI

In [13]:
# Compare row counts and total quantities for the selected invoices.

invoice_summary = (
    invoice_review
    .groupby("Invoice")
    .agg(
        row_count=("Invoice", "size"),
        distinct_products=("StockCode", "nunique"),
        total_quantity=("Quantity", "sum"),
        recorded_line_value=("LineValue", "sum")
    )
    .reset_index()
)

display(invoice_summary)

,Invoice,row_count,distinct_products,total_quantity,recorded_line_value
0,537144,210,95,422,686.56
1,537781,322,127,582,1027.20
2,555524,76,39,265,610.77


In [14]:
invoice_duplicate_review = (
    invoice_review
    .groupby(
        [
            "Invoice",
            "StockCode",
            "Description",
            "Quantity",
            "Price"
        ],
        dropna=False
    )
    .agg(
        occurrences=("LineValue", "size"),
        line_value_per_row=("LineValue", "first")
    )
    .reset_index()
)

invoice_duplicate_review = invoice_duplicate_review[
    invoice_duplicate_review["occurrences"] > 1
].copy()

invoice_duplicate_review["combined_line_value"] = (
    invoice_duplicate_review["occurrences"]
    * invoice_duplicate_review["line_value_per_row"]
)

display(
    invoice_duplicate_review.sort_values(
        ["Invoice", "occurrences"],
        ascending=[True, False]
    )
)

,Invoice,StockCode,Description,Quantity,Price,occurrences,line_value_per_row,combined_line_value
24,537144,21882,SKULLS TAPE,1,0.65,6,0.65,3.9
31,537144,22086,PAPER CHAIN KIT 50'S CHRISTMAS,1,2.95,6,2.95,17.7
32,537144,22095,LADS ONLY TISSUE BOX,1,1.25,4,1.25,5.0
33,537144,22100,SKULLS SQUARE TISSUE BOX,1,1.25,4,1.25,5.0
42,537144,22294,HEART FILIGREE DOVE SMALL,1,1.25,4,1.25,5.0
...,...,...,...,...,...,...,...,...
248,555524,22697,GREEN REGENCY TEACUP AND SAUCER,1,2.95,12,2.95,35.4
233,555524,21155,RED RETROSPOT PEG BAG,1,2.55,2,2.55,5.1
253,555524,22776,SWEETHEART CAKESTAND 3 TIER,1,9.95,2,9.95,19.9
262,555524,23170,REGENCY TEA PLATE ROSES,12,1.65,2,19.80,39.6


In [15]:
from pathlib import Path
import pandas as pd

project_root = Path(
    r"D:\ai_powered_data_analyst\Retail_Analytics"
)

report_dir = project_root / "reports"
report_dir.mkdir(parents=True, exist_ok=True)

# Summary metrics from our actual dataset
quality_report = pd.DataFrame([
    {
        "Check": "Original row count",
        "Result": len(audit),
        "Interpretation": "Preserved source rows"
    },
    {
        "Check": "Exact duplicate rows removable",
        "Result": int(audit.duplicated().sum()),
        "Interpretation": "Candidates for review; not automatically removed"
    },
    {
        "Check": "Distinct repeated row patterns",
        "Result": len(duplicate_groups),
        "Interpretation": "Unique combinations occurring more than once"
    },
    {
        "Check": "Rows in repeated groups",
        "Result": int(duplicate_groups["OccurrenceCount"].sum()),
        "Interpretation": "Includes first and subsequent occurrences"
    },
    {
        "Check": "Missing customer IDs",
        "Result": int(audit["Customer ID"].isna().sum()),
        "Interpretation": "Retain as missing; exclude from customer-level analysis"
    },
    {
        "Check": "Missing descriptions",
        "Result": int(audit["Description"].isna().sum()),
        "Interpretation": "Investigate; do not automatically delete"
    },
    {
        "Check": "Zero-price rows",
        "Result": int(audit["Price"].eq(0).sum()),
        "Interpretation": "Investigate separately"
    },
    {
        "Check": "Negative-quantity rows",
        "Result": int(audit["Quantity"].lt(0).sum()),
        "Interpretation": "Review as potential returns or adjustments"
    },
])

report_path = report_dir / "data_quality_audit.csv"
quality_report.to_csv(report_path, index=False)

print("Report saved to:", report_path)
display(quality_report)

Report saved to: D:\ai_powered_data_analyst\Retail_Analytics\reports\data_quality_audit.csv


,Check,Result,Interpretation
0,Original row count,1067371,Preserved source rows
1,Exact duplicate rows removable,34335,Candidates for review; not automatically removed
2,Distinct repeated row patterns,32907,Unique combinations occurring more than once
3,Rows in repeated groups,67242,Includes first and subsequent occurrences
4,Missing customer IDs,243007,Retain as missing; exclude from customer-level...
5,Missing descriptions,4382,Investigate; do not automatically delete
6,Zero-price rows,6202,Investigate separately
7,Negative-quantity rows,22950,Review as potential returns or adjustments


In [16]:

# Create a separate working copy.
# Keep df_full and audit unchanged.
retail = audit.copy()

# 1. Cancellation invoice indicator
retail["IsCancellationInvoice"] = (
    retail["Invoice"].astype(str).str.strip().str.startswith("C")
)

# 2. Negative quantity indicator
retail["IsNegativeQuantity"] = retail["Quantity"] < 0

# 3. Zero or negative unit price
retail["IsZeroPrice"] = retail["Price"] == 0
retail["IsNegativePrice"] = retail["Price"] < 0

# 4. Missing information
retail["MissingCustomerID"] = retail["Customer ID"].isna()
retail["MissingDescription"] = retail["Description"].isna()

# 5. Exact duplicate candidate indicator
# True means this row is a second or subsequent occurrence.
retail["IsExactDuplicateCandidate"] = audit.duplicated(
    subset=[
        "Invoice",
        "StockCode",
        "Description",
        "Quantity",
        "InvoiceDate",
        "Price",
        "Customer ID",
        "Country"
    ],
    keep="first"
)

# 6. Special descriptions
retail["IsSpecialDescription"] = (
    retail["Description_clean"].str.contains(
        r"adjust bad debt|manual|bank charge|amazon fee",
        regex=True,
        na=False
    )
)

# 7. Calculate recorded line value
retail["LineValue"] = retail["Quantity"] * retail["Price"]

print("Classified dataset shape:", retail.shape)
print("Original rows preserved:", len(retail) == len(df_full))
display(retail.head())


Classified dataset shape: (1067371, 18)
Original rows preserved: True


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Description_clean,LineValue,IsCancellationInvoice,IsNegativeQuantity,IsZeroPrice,IsNegativePrice,MissingCustomerID,MissingDescription,IsExactDuplicateCandidate,IsSpecialDescription
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,15cm christmas glass ball 20 lights,83.4,False,False,False,False,False,False,False,False
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,pink cherry lights,81.0,False,False,False,False,False,False,False,False
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,white cherry lights,81.0,False,False,False,False,False,False,False,False
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,"record frame 7"" single size",100.8,False,False,False,False,False,False,False,False
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,strawberry ceramic trinket box,30.0,False,False,False,False,False,False,False,False


In [17]:

flag_columns = [
    "IsCancellationInvoice",
    "IsNegativeQuantity",
    "IsZeroPrice",
    "IsNegativePrice",
    "MissingCustomerID",
    "MissingDescription",
    "IsExactDuplicateCandidate",
    "IsSpecialDescription"
]

flag_summary = pd.DataFrame({
    "Flag": flag_columns,
    "Rows_flagged": [
        int(retail[col].sum()) for col in flag_columns
    ]
})

flag_summary["Percentage_of_rows"] = (
    flag_summary["Rows_flagged"] / len(retail) * 100
).round(2)

display(flag_summary)

print("Rows in retail:", len(retail))
print("Rows in df_full:", len(df_full))
print("Row counts match:", len(retail) == len(df_full))


,Flag,Rows_flagged,Percentage_of_rows
0,IsCancellationInvoice,19494,1.83
1,IsNegativeQuantity,22950,2.15
2,IsZeroPrice,6202,0.58
3,IsNegativePrice,5,0.00
4,MissingCustomerID,243007,22.77
5,MissingDescription,4382,0.41
6,IsExactDuplicateCandidate,34335,3.22
7,IsSpecialDescription,1577,0.15


Rows in retail: 1067371
Rows in df_full: 1067371
Row counts match: True


In [18]:

# Begin with an explicit, provisional classification.
retail["TransactionType"] = "Review"

# Identify special records by description.
retail.loc[
    retail["IsSpecialDescription"],
    "TransactionType"
] = "Special / non-merchandise candidate"

# Identify cancellation invoices not already classified as special.
retail.loc[
    retail["IsCancellationInvoice"]
    & ~retail["IsSpecialDescription"],
    "TransactionType"
] = "Cancellation invoice"

# Identify negative quantities on non-cancellation invoices.
retail.loc[
    ~retail["IsCancellationInvoice"]
    & retail["IsNegativeQuantity"]
    & ~retail["IsSpecialDescription"],
    "TransactionType"
] = "Negative quantity / review"

# Identify remaining records with non-positive prices.
retail.loc[
    ~retail["IsCancellationInvoice"]
    & ~retail["IsNegativeQuantity"]
    & (retail["Price"] <= 0)
    & ~retail["IsSpecialDescription"],
    "TransactionType"
] = "Non-positive price / review"

# Everything remaining is a merchandise candidate,
# not yet a verified sale.
retail.loc[
    retail["TransactionType"] == "Review",
    "TransactionType"
] = "Merchandise candidate"

# Summarize categories using the recorded line value.
transaction_summary = (
    retail.groupby("TransactionType", dropna=False)
    .agg(
        row_count=("Invoice", "size"),
        unique_invoices=("Invoice", "nunique"),
        recorded_line_value=("LineValue", "sum")
    )
    .reset_index()
)

transaction_summary["recorded_line_value"] = (
    transaction_summary["recorded_line_value"].round(2)
)

transaction_summary["percentage_of_rows"] = (
    transaction_summary["row_count"] / len(retail) * 100
).round(2)

display(
    transaction_summary.sort_values(
        "row_count",
        ascending=False
    )
)

print("Total classified rows:", transaction_summary["row_count"].sum())
print("Original rows:", len(df_full))


,TransactionType,row_count,unique_invoices,recorded_line_value,percentage_of_rows
1,Merchandise candidate,1040749,39732,20586257.81,97.51
0,Cancellation invoice,18850,7784,-772285.68,1.77
2,Negative quantity / review,3457,3393,0.00,0.32
3,Non-positive price / review,2738,1979,0.00,0.26
4,Special / non-merchandise candidate,1577,1425,-526721.56,0.15


Total classified rows: 1067371
Original rows: 1067371


In [19]:

value_check = (
    retail.groupby("TransactionType")
    .agg(
        row_count=("LineValue", "size"),
        positive_value_rows=("LineValue", lambda s: (s > 0).sum()),
        negative_value_rows=("LineValue", lambda s: (s < 0).sum()),
        zero_value_rows=("LineValue", lambda s: (s == 0).sum()),
        total_recorded_value=("LineValue", "sum"),
        sum_absolute_values=("LineValue", lambda s: s.abs().sum())
    )
    .reset_index()
)

money_columns = [
    "total_recorded_value",
    "sum_absolute_values"
]

value_check[money_columns] = value_check[money_columns].round(2)

display(value_check)


,TransactionType,row_count,positive_value_rows,negative_value_rows,zero_value_rows,total_recorded_value,sum_absolute_values
0,Cancellation invoice,18850,0,18850,0,-772285.68,772285.68
1,Merchandise candidate,1040749,1040749,0,0,20586257.81,20586257.81
2,Negative quantity / review,3457,0,0,3457,0.00,0.00
3,Non-positive price / review,2738,0,0,2738,0.00,0.00
4,Special / non-merchandise candidate,1577,922,648,7,-526721.56,1300142.22


In [20]:

for _, row in value_check.iterrows():
    print(f"\nTransaction type: {row['TransactionType']}")
    print(f"Rows: {row['row_count']:,}")
    print(f"Positive-value rows: {row['positive_value_rows']:,}")
    print(f"Negative-value rows: {row['negative_value_rows']:,}")
    print(f"Zero-value rows: {row['zero_value_rows']:,}")
    print(f"Net recorded line value: £{row['total_recorded_value']:,.2f}")
    print(f"Sum of absolute line values: £{row['sum_absolute_values']:,.2f}")



Transaction type: Cancellation invoice
Rows: 18,850
Positive-value rows: 0
Negative-value rows: 18,850
Zero-value rows: 0
Net recorded line value: £-772,285.68
Sum of absolute line values: £772,285.68

Transaction type: Merchandise candidate
Rows: 1,040,749
Positive-value rows: 1,040,749
Negative-value rows: 0
Zero-value rows: 0
Net recorded line value: £20,586,257.81
Sum of absolute line values: £20,586,257.81

Transaction type: Negative quantity / review
Rows: 3,457
Positive-value rows: 0
Negative-value rows: 0
Zero-value rows: 3,457
Net recorded line value: £0.00
Sum of absolute line values: £0.00

Transaction type: Non-positive price / review
Rows: 2,738
Positive-value rows: 0
Negative-value rows: 0
Zero-value rows: 2,738
Net recorded line value: £0.00
Sum of absolute line values: £0.00

Transaction type: Special / non-merchandise candidate
Rows: 1,577
Positive-value rows: 922
Negative-value rows: 648
Zero-value rows: 7
Net recorded line value: £-526,721.56
Sum of absolute line va

In [21]:
# Inspect sample rows
display(
    retail.loc[
        retail["TransactionType"] == "Negative quantity / review",
        ["Invoice", "StockCode", "Description", "Quantity", "Price", "LineValue"]
    ].head(15)
)

# Check quantity, price, and line-value statistics
display(
    retail.loc[
        retail["TransactionType"] == "Negative quantity / review",
        ["Quantity", "Price", "LineValue"]
    ].describe()
)

# Verify the calculations directly
negative_review = retail[
    retail["TransactionType"] == "Negative quantity / review"
]

print("Rows:", len(negative_review))
print("Negative quantities:", (negative_review["Quantity"] < 0).sum())
print("Zero prices:", (negative_review["Price"] == 0).sum())
print("Negative prices:", (negative_review["Price"] < 0).sum())
print("Net line value:", negative_review["LineValue"].sum())
print("Sum of absolute line values:", negative_review["LineValue"].abs().sum())
print(
    "Rows where LineValue differs from Quantity × Price:",
    (
        negative_review["LineValue"]
        != negative_review["Quantity"] * negative_review["Price"]
    ).sum()
)

,Invoice,StockCode,Description,Quantity,Price,LineValue
263,489464,21733,85123a mixed,-96,0.0,-0.0
283,489463,71477,short,-240,0.0,-0.0
284,489467,85123A,21733 mixed,-192,0.0,-0.0
470,489521,21646,NaN,-50,0.0,-0.0
3114,489655,20683,NaN,-44,0.0,-0.0
3162,489660,35956,lost,-1043,0.0,-0.0
3168,489663,35605A,damages,-117,0.0,-0.0
4296,489806,18010,NaN,-770,0.0,-0.0
4538,489820,21133,invcd as 84879?,-720,0.0,-0.0
4566,489821,85049G,NaN,-240,0.0,-0.0


,Quantity,Price,LineValue
count,3457.000000,3457.0,3457.0
mean,-165.775239,0.0,0.0
std,673.108503,0.0,0.0
min,-9600.000000,0.0,-0.0
25%,-82.000000,0.0,0.0
50%,-26.000000,0.0,0.0
75%,-8.000000,0.0,0.0
max,-1.000000,0.0,-0.0


Rows: 3457
Negative quantities: 3457
Zero prices: 3457
Negative prices: 0
Net line value: 0.0
Sum of absolute line values: 0.0
Rows where LineValue differs from Quantity × Price: 0


In [22]:
negative_review = retail[
    retail["TransactionType"] == "Negative quantity / review"
].copy()

# Most common descriptions
print("Top descriptions:")
display(
    negative_review["Description"]
    .fillna("[Missing]")
    .value_counts()
    .head(25)
    .rename_axis("Description")
    .reset_index(name="RowCount")
)

# Invoice prefix and customer-ID breakdown
negative_review["InvoicePrefix"] = (
    negative_review["Invoice"].astype(str).str.startswith("C")
)

display(
    negative_review.groupby("InvoicePrefix", dropna=False)
    .agg(
        rows=("Invoice", "size"),
        unique_invoices=("Invoice", "nunique"),
        customers_missing=("Customer ID", lambda s: s.isna().sum()),
        distinct_customers=("Customer ID", "nunique")
    )
    .reset_index()
)

# Check the original records for a few invoices
display(
    negative_review[
        ["Invoice", "StockCode", "Description", "Quantity",
         "InvoiceDate", "Price", "Customer ID", "Country"]
    ].head(30)
)

Top descriptions:


,Description,RowCount
0,[Missing],2689
1,check,123
2,damages,84
3,?,83
4,damaged,78
5,missing,27
6,sold as set on dotcom,20
7,Damaged,17
8,"Unsaleable, destroyed.",9
9,smashed,9


,InvoicePrefix,rows,unique_invoices,customers_missing,distinct_customers
0,False,3457,3393,3457,0


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
263,489464,21733,85123a mixed,-96,2009-12-01 10:52:00,0.0,NaN,United Kingdom
283,489463,71477,short,-240,2009-12-01 10:52:00,0.0,NaN,United Kingdom
284,489467,85123A,21733 mixed,-192,2009-12-01 10:53:00,0.0,NaN,United Kingdom
470,489521,21646,NaN,-50,2009-12-01 11:44:00,0.0,NaN,United Kingdom
3114,489655,20683,NaN,-44,2009-12-01 17:26:00,0.0,NaN,United Kingdom
3162,489660,35956,lost,-1043,2009-12-01 17:43:00,0.0,NaN,United Kingdom
3168,489663,35605A,damages,-117,2009-12-01 18:02:00,0.0,NaN,United Kingdom
4296,489806,18010,NaN,-770,2009-12-02 12:42:00,0.0,NaN,United Kingdom
4538,489820,21133,invcd as 84879?,-720,2009-12-02 13:23:00,0.0,NaN,United Kingdom
4566,489821,85049G,NaN,-240,2009-12-02 13:25:00,0.0,NaN,United Kingdom


In [23]:
import numpy as np

negative_review = retail[
    retail["TransactionType"] == "Negative quantity / review"
].copy()

description = (
    negative_review["Description"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)

conditions = [
    description.eq(""),
    description.str.contains(
        r"damage|damaged|smashed|crushed|wet|destroyed|unsaleable",
        regex=True,
        na=False
    ),
    description.str.contains(
        r"lost|missing|mia",
        regex=True,
        na=False
    ),
    description.str.contains(
        r"dotcom|ebay|sold as|given away",
        regex=True,
        na=False
    ),
    description.str.contains(
        r"check|counted|^\?+$",
        regex=True,
        na=False
    ),
]

labels = [
    "Missing description",
    "Damage-related wording",
    "Lost/missing wording",
    "Sales/channel/other handling wording",
    "Check/count/uncertain wording",
]

negative_review["DescriptionGroup"] = np.select(
    conditions,
    labels,
    default="Other / needs review"
)

description_summary = (
    negative_review.groupby("DescriptionGroup")
    .agg(
        rows=("Invoice", "size"),
        unique_invoices=("Invoice", "nunique"),
        total_quantity=("Quantity", "sum"),
        recorded_line_value=("LineValue", "sum")
    )
    .reset_index()
    .sort_values("rows", ascending=False)
)

description_summary["row_percentage"] = (
    description_summary["rows"] / len(negative_review) * 100
).round(2)

display(description_summary)

,DescriptionGroup,rows,unique_invoices,total_quantity,recorded_line_value,row_percentage
3,Missing description,2689,2633,-258288,0.0,77.78
1,Damage-related wording,264,263,-60533,0.0,7.64
0,Check/count/uncertain wording,230,226,-54809,0.0,6.65
4,Other / needs review,139,138,-100747,0.0,4.02
5,Sales/channel/other handling wording,81,79,-69148,0.0,2.34
2,Lost/missing wording,54,54,-29560,0.0,1.56


In [24]:
print("Rows classified:", description_summary["rows"].sum())
print("Original rows:", len(negative_review))
print(
    "Quantity reconciles:",
    description_summary["total_quantity"].sum()
    == negative_review["Quantity"].sum()
)
print(
    "Recorded value reconciles:",
    round(description_summary["recorded_line_value"].sum(), 2)
    == round(negative_review["LineValue"].sum(), 2)
)

Rows classified: 3457
Original rows: 3457
Quantity reconciles: True
Recorded value reconciles: True


In [25]:
# Summarise the current transaction classifications
classification_summary = (
    retail.groupby("TransactionType", dropna=False)
    .agg(
        row_count=("Invoice", "size"),
        invoice_count=("Invoice", "nunique"),
        customer_ids_available=("Customer ID", "count"),
        total_quantity=("Quantity", "sum"),
        recorded_line_value=("LineValue", "sum")
    )
    .reset_index()
)

classification_summary["row_percentage"] = (
    classification_summary["row_count"]
    / len(retail) * 100
).round(2)

classification_summary["recorded_line_value"] = (
    classification_summary["recorded_line_value"].round(2)
)

display(classification_summary)

,TransactionType,row_count,invoice_count,customer_ids_available,total_quantity,recorded_line_value,row_percentage
0,Cancellation invoice,18850,7784,18339,-485426,-772285.68,1.77
1,Merchandise candidate,1040749,39732,804808,11410210,20586257.81,97.51
2,Negative quantity / review,3457,3393,0,-573085,0.00,0.32
3,Non-positive price / review,2738,1979,64,252252,0.00,0.26
4,Special / non-merchandise candidate,1577,1425,1153,4541,-526721.56,0.15


In [26]:
report_path = (
    r"D:\ai_powered_data_analyst\Retail_Analytics"
    r"\reports\transaction_classification_summary.csv"
)

classification_summary.to_csv(report_path, index=False)

print("Saved:", report_path)

Saved: D:\ai_powered_data_analyst\Retail_Analytics\reports\transaction_classification_summary.csv


In [27]:
analysis = retail.copy()

# 1. Merchandise performance candidate
analysis["Include_Merchandise_Analysis"] = (
    analysis["TransactionType"] == "Merchandise candidate"
)

# 2. Customer-level analysis candidate
analysis["Include_Customer_Analysis"] = (
    analysis["Include_Merchandise_Analysis"]
    & analysis["Customer ID"].notna()
    & analysis["InvoiceDate"].notna()
)

# 3. Records requiring separate review
analysis["Requires_Review"] = (
    analysis["TransactionType"] != "Merchandise candidate"
)

# Keep duplicate candidates visible; do not remove them yet
analysis["Duplicate_Sensitivity_Flag"] = (
    analysis["IsExactDuplicateCandidate"]
)

print("Original retail rows:", len(retail))
print("Analysis rows:", len(analysis))
print(
    "Merchandise-analysis candidates:",
    analysis["Include_Merchandise_Analysis"].sum()
)
print(
    "Customer-analysis candidates:",
    analysis["Include_Customer_Analysis"].sum()
)
print("Rows requiring review:", analysis["Requires_Review"].sum())
print(
    "Duplicate candidates:",
    analysis["Duplicate_Sensitivity_Flag"].sum()
)

assert len(analysis) == len(retail)

Original retail rows: 1067371
Analysis rows: 1067371
Merchandise-analysis candidates: 1040749
Customer-analysis candidates: 804808
Rows requiring review: 26622
Duplicate candidates: 34335


In [28]:
# Compare duplicate candidates across the proposed analysis populations
duplicate_audit = (
    analysis.groupby("Duplicate_Sensitivity_Flag")
    .agg(
        row_count=("Invoice", "size"),
        merchandise_candidates=("Include_Merchandise_Analysis", "sum"),
        customer_analysis_candidates=("Include_Customer_Analysis", "sum"),
        recorded_line_value=("LineValue", "sum"),
        unique_invoices=("Invoice", "nunique")
    )
    .reset_index()
)

duplicate_audit["recorded_line_value"] = (
    duplicate_audit["recorded_line_value"].round(2)
)

display(duplicate_audit)

# Check whether duplicate candidates occur in the merchandise population
duplicate_merchandise = analysis[
    analysis["Duplicate_Sensitivity_Flag"]
    & analysis["Include_Merchandise_Analysis"]
]

print(
    "Duplicate candidates within merchandise population:",
    len(duplicate_merchandise)
)
print(
    "Their recorded line value:",
    round(duplicate_merchandise["LineValue"].sum(), 2)
)

# Inspect the most frequently repeated merchandise lines
repeat_patterns = (
    analysis.loc[
        analysis["Include_Merchandise_Analysis"],
        [
            "Invoice", "StockCode", "Description", "Quantity",
            "InvoiceDate", "Price", "Customer ID", "Country"
        ]
    ]
    .groupby(
        [
            "Invoice", "StockCode", "Description", "Quantity",
            "InvoiceDate", "Price", "Customer ID", "Country"
        ],
        dropna=False
    )
    .size()
    .rename("Occurrences")
    .reset_index()
)

display(
    repeat_patterns[repeat_patterns["Occurrences"] > 1]
    .sort_values("Occurrences", ascending=False)
    .head(20)
)

,Duplicate_Sensitivity_Flag,row_count,merchandise_candidates,customer_analysis_candidates,recorded_line_value,unique_invoices
0,False,1033036,1007024,778713,18855533.70,53628
1,True,34335,33725,26095,431716.87,5391


Duplicate candidates within merchandise population: 33725
Their recorded line value: 481287.75


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Occurrences
688611,555524,22698,PINK REGENCY TEACUP AND SAUCER,1,2011-06-05 11:37:00,2.95,16923.0,United Kingdom,20
688610,555524,22697,GREEN REGENCY TEACUP AND SAUCER,1,2011-06-05 11:37:00,2.95,16923.0,United Kingdom,12
492164,537224,70007,HI TEC ALPINE HAND WARMER,1,2010-12-05 16:24:00,1.65,13174.0,United Kingdom,10
888713,572861,22775,PURPLE DRAWERKNOB ACRYLIC EDWARDIAN,12,2011-10-26 12:46:00,1.25,14102.0,United Kingdom,8
74195,496431,84826,ASSTD DESIGN 3D PAPER STICKERS,1,2010-02-01 12:30:00,0.85,16415.0,United Kingdom,6
489722,537042,21579,LOLITA DESIGN COTTON TOTE BAG,1,2010-12-05 10:45:00,2.25,13838.0,United Kingdom,6
488017,536874,22866,HAND WARMER SCOTTY DOG DESIGN,1,2010-12-03 11:35:00,2.10,16891.0,United Kingdom,6
530333,540524,21756,BATH BUILDING BLOCK WORD,1,2011-01-09 12:53:00,5.95,16735.0,United Kingdom,6
499777,537781,22898,CHILDRENS APRON APPLES DESIGN,1,2010-12-08 12:46:00,1.95,17341.0,United Kingdom,6
539438,541266,21754,HOME BUILDING BLOCK WORD,1,2011-01-16 16:25:00,5.95,15673.0,United Kingdom,6


In [29]:
# Primary merchandise population: preserve all candidate rows
merchandise_all = analysis.loc[
    analysis["Include_Merchandise_Analysis"]
].copy()

# Sensitivity version: exclude only rows flagged as later exact matches
merchandise_without_duplicates = merchandise_all.loc[
    ~merchandise_all["Duplicate_Sensitivity_Flag"]
].copy()


def merchandise_metrics(data, version):
    return {
        "Version": version,
        "Rows": len(data),
        "Unique invoices": data["Invoice"].nunique(),
        "Unique products": data["StockCode"].nunique(),
        "Customers identified": data["Customer ID"].nunique(),
        "Rows with missing customer ID": data["Customer ID"].isna().sum(),
        "Recorded line value": data["LineValue"].sum(),
        "Total quantity": data["Quantity"].sum(),
        "Duplicate candidate rows retained":
            data["Duplicate_Sensitivity_Flag"].sum()
    }


sensitivity_comparison = pd.DataFrame([
    merchandise_metrics(merchandise_all, "Primary: retain all"),
    merchandise_metrics(
        merchandise_without_duplicates,
        "Sensitivity: exclude exact matches"
    )
])

sensitivity_comparison["Recorded line value"] = (
    sensitivity_comparison["Recorded line value"].round(2)
)

display(sensitivity_comparison)

print(
    "Difference in recorded line value:",
    round(
        merchandise_all["LineValue"].sum()
        - merchandise_without_duplicates["LineValue"].sum(),
        2
    )
)

print(
    "Rows excluded in sensitivity version:",
    len(merchandise_all) - len(merchandise_without_duplicates)
)

,Version,Rows,Unique invoices,Unique products,Customers identified,Rows with missing customer ID,Recorded line value,Total quantity,Duplicate candidate rows retained
0,Primary: retain all,1040749,39732,4912,5863,235941,20586257.81,11410210,33725
1,Sensitivity: exclude exact matches,1007024,39732,4912,5863,228311,20104970.06,11195476,0


Difference in recorded line value: 481287.75
Rows excluded in sensitivity version: 33725


In [30]:
for name, data in [
    ("PRIMARY — RETAIN ALL", merchandise_all),
    ("SENSITIVITY — EXCLUDE FLAGGED ROWS",
     merchandise_without_duplicates)
]:
    print(f"\n{'=' * 45}")
    print(name)
    print("=" * 45)

    print("Rows:", f"{len(data):,}")
    print("Unique invoices:", f"{data['Invoice'].nunique():,}")
    print("Unique products:", f"{data['StockCode'].nunique():,}")
    print("Identified customers:", f"{data['Customer ID'].nunique():,}")
    print(
        "Rows missing customer ID:",
        f"{data['Customer ID'].isna().sum():,}"
    )
    print("Total quantity:", f"{data['Quantity'].sum():,.0f}")
    print("Recorded line value:", f"£{data['LineValue'].sum():,.2f}")
    print(
        "Duplicate candidates retained:",
        f"{data['Duplicate_Sensitivity_Flag'].sum():,}"
    )


PRIMARY — RETAIN ALL
Rows: 1,040,749
Unique invoices: 39,732
Unique products: 4,912
Identified customers: 5,863
Rows missing customer ID: 235,941
Total quantity: 11,410,210
Recorded line value: £20,586,257.81
Duplicate candidates retained: 33,725

SENSITIVITY — EXCLUDE FLAGGED ROWS
Rows: 1,007,024
Unique invoices: 39,732
Unique products: 4,912
Identified customers: 5,863
Rows missing customer ID: 228,311
Total quantity: 11,195,476
Recorded line value: £20,104,970.06
Duplicate candidates retained: 0


In [31]:
from pathlib import Path

project_root = Path(r"D:\ai_powered_data_analyst\Retail_Analytics")
processed_dir = project_root / "data" / "processed"
report_dir = project_root / "reports"

processed_dir.mkdir(parents=True, exist_ok=True)
report_dir.mkdir(parents=True, exist_ok=True)

# 1. Full classified dataset — preserves every original row
analysis.to_csv(
    processed_dir / "retail_classified_full.csv",
    index=False
)

# 2. Primary merchandise-analysis dataset — retains duplicate candidates
merchandise_all.to_csv(
    processed_dir / "retail_merchandise_primary.csv",
    index=False
)

# 3. Sensitivity dataset — excludes flagged exact-match candidates
merchandise_without_duplicates.to_csv(
    processed_dir / "retail_merchandise_sensitivity.csv",
    index=False
)

# 4. Customer-level analysis candidates
customer_analysis = analysis.loc[
    analysis["Include_Customer_Analysis"]
].copy()

customer_analysis.to_csv(
    processed_dir / "retail_customer_analysis_candidates.csv",
    index=False
)

print("Exports completed.")
print("Full classified dataset:", len(analysis))
print("Primary merchandise dataset:", len(merchandise_all))
print("Sensitivity dataset:", len(merchandise_without_duplicates))
print("Customer-analysis candidates:", len(customer_analysis))

Exports completed.
Full classified dataset: 1067371
Primary merchandise dataset: 1040749
Sensitivity dataset: 1007024
Customer-analysis candidates: 804808


In [33]:
import pandas as pd

files_to_check = {
    "Full classified": processed_dir / "retail_classified_full.csv",
    "Primary merchandise": processed_dir / "retail_merchandise_primary.csv",
    "Sensitivity merchandise": processed_dir / "retail_merchandise_sensitivity.csv",
    "Customer candidates": processed_dir / "retail_customer_analysis_candidates.csv",
}

for name, path in files_to_check.items():
    check_df = pd.read_csv(path, low_memory=False)
    print(f"{name}: {len(check_df):,} rows | exists={path.exists()}")

Full classified: 1,067,371 rows | exists=True
Primary merchandise: 1,040,749 rows | exists=True
Sensitivity merchandise: 1,007,024 rows | exists=True
Customer candidates: 804,808 rows | exists=True


In [34]:
import pandas as pd

# Reconciliation checks
assert len(analysis) == len(retail) == 1_067_371
assert len(merchandise_all) == 1_040_749
assert len(merchandise_without_duplicates) == 1_007_024
assert len(customer_analysis) == 804_808

# Check exported row counts and recorded line values
export_checks = [
    (
        "Full classified",
        processed_dir / "retail_classified_full.csv",
        analysis
    ),
    (
        "Primary merchandise",
        processed_dir / "retail_merchandise_primary.csv",
        merchandise_all
    ),
    (
        "Sensitivity merchandise",
        processed_dir / "retail_merchandise_sensitivity.csv",
        merchandise_without_duplicates
    ),
    (
        "Customer candidates",
        processed_dir / "retail_customer_analysis_candidates.csv",
        customer_analysis
    ),
]

for name, path, source_df in export_checks:
    exported = pd.read_csv(path, low_memory=False)

    source_value = round(source_df["LineValue"].sum(), 2)
    export_value = round(exported["LineValue"].sum(), 2)

    print(f"\n{name}")
    print("Row counts match:", len(exported) == len(source_df))
    print("Column names match:", list(exported.columns) == list(source_df.columns))
    print(f"Source line value:  £{source_value:,.2f}")
    print(f"Export line value:  £{export_value:,.2f}")
    print("Line values reconcile:", source_value == export_value)


Full classified
Row counts match: True
Column names match: True
Source line value:  £19,287,250.57
Export line value:  £19,287,250.57
Line values reconcile: True

Primary merchandise
Row counts match: True
Column names match: True
Source line value:  £20,586,257.81
Export line value:  £20,586,257.81
Line values reconcile: True

Sensitivity merchandise
Row counts match: True
Column names match: True
Source line value:  £20,104,970.06
Export line value:  £20,104,970.06
Line values reconcile: True

Customer candidates
Row counts match: True
Column names match: True
Source line value:  £17,590,623.61
Export line value:  £17,590,623.61
Line values reconcile: True


In [35]:
sql_export_path = (
    processed_dir / "retail_classified_full_sql.csv"
)

sql_df = analysis.copy()

sql_df.columns = (
    sql_df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
)

sql_df.to_csv(sql_export_path, index=False)

print("Export:", sql_export_path)
print("Rows:", len(sql_df))
print("Columns:")
print(sql_df.columns.tolist())

Export: D:\ai_powered_data_analyst\Retail_Analytics\data\processed\retail_classified_full_sql.csv
Rows: 1067371
Columns:
['invoice', 'stockcode', 'description', 'quantity', 'invoicedate', 'price', 'customer_id', 'country', 'description_clean', 'linevalue', 'iscancellationinvoice', 'isnegativequantity', 'iszeroprice', 'isnegativeprice', 'missingcustomerid', 'missingdescription', 'isexactduplicatecandidate', 'isspecialdescription', 'transactiontype', 'include_merchandise_analysis', 'include_customer_analysis', 'requires_review', 'duplicate_sensitivity_flag']


In [36]:

# Final Data Quality Validation — Step 1
# This cell only checks existing objects. It does not modify your data.

required_objects = [
    "audit",
    "retail",
    "analysis",
    "customer_analysis"
]

for name in required_objects:
    if name in globals():
        obj = globals()[name]
        print(f"{name}: {len(obj):,} rows × {len(obj.columns):,} columns")
    else:
        print(f"{name}: NOT FOUND")


audit: 1,067,371 rows × 11 columns
retail: 1,067,371 rows × 19 columns
analysis: 1,067,371 rows × 23 columns
customer_analysis: 804,808 rows × 23 columns


In [67]:
# Final Data Quality Validation — Step 2
# Missing-value audit across every column in analysis

import pandas as pd

missing_audit = pd.DataFrame({
    "column": analysis.columns,
    "data_type": [str(dtype) for dtype in analysis.dtypes],
    "missing_count": analysis.isna().sum().values,
    "missing_percentage": (
        analysis.isna().mean().values * 100
    ).round(2)
})

missing_audit = missing_audit.sort_values(
    by="missing_count",
    ascending=False
).reset_index(drop=True)

print("Total rows:", f"{len(analysis):,}")
print("Total columns:", len(analysis.columns))
print(
    "Total missing cells:",
    f"{int(analysis.isna().sum().sum()):,}"
)

display(missing_audit)


Total rows: 1,067,371
Total columns: 23
Total missing cells: 247,389


,column,data_type,missing_count,missing_percentage
0,Customer ID,float64,243007,22.77
1,Description,object,4382,0.41
2,Invoice,object,0,0.00
3,IsNegativePrice,bool,0,0.00
4,Requires_Review,bool,0,0.00
5,Include_Customer_Analysis,bool,0,0.00
6,Include_Merchandise_Analysis,bool,0,0.00
7,TransactionType,object,0,0.00
8,IsSpecialDescription,bool,0,0.00
9,IsExactDuplicateCandidate,bool,0,0.00


In [69]:

# Final Data Quality Validation — Step 3
# Verify duplicate flags against the original transaction fields.

duplicate_columns = [
    "Invoice",
    "StockCode",
    "Description",
    "Quantity",
    "InvoiceDate",
    "Price",
    "Customer ID",
    "Country"
]

# Independently recompute exact duplicate candidates
recomputed_duplicates = analysis.duplicated(
    subset=duplicate_columns,
    keep="first"
)

# Compare recomputed results with the existing flags
flag_matches = (
    recomputed_duplicates
    == analysis["IsExactDuplicateCandidate"]
).all()

sensitivity_flag_matches = (
    analysis["IsExactDuplicateCandidate"]
    == analysis["Duplicate_Sensitivity_Flag"]
).all()

print("Total rows:", f"{len(analysis):,}")
print(
    "Recomputed duplicate candidates:",
    f"{int(recomputed_duplicates.sum()):,}"
)
print(
    "Existing duplicate flag count:",
    f"{int(analysis['IsExactDuplicateCandidate'].sum()):,}"
)
print(
    "Duplicate_Sensitivity_Flag count:",
    f"{int(analysis['Duplicate_Sensitivity_Flag'].sum()):,}"
)
print("Recomputed flags match existing flags:", flag_matches)
print(
    "Duplicate sensitivity flags match:",
    sensitivity_flag_matches
)
print(
    "Rows removed during this check: 0"
)


Total rows: 1,067,371
Recomputed duplicate candidates: 34,335
Existing duplicate flag count: 34,335
Duplicate_Sensitivity_Flag count: 34,335
Recomputed flags match existing flags: True
Duplicate sensitivity flags match: True
Rows removed during this check: 0


In [71]:

# Final Data Quality Validation — Step 4
# Check numeric integrity, dates, and classification flags.

import numpy as np
import pandas as pd

# 1. Check for non-finite numeric values
numeric_checks = {}

for col in ["Quantity", "Price", "LineValue"]:
    values = pd.to_numeric(analysis[col], errors="coerce")
    numeric_checks[col] = {
        "non_numeric": int(values.isna().sum()),
        "non_finite": int((~np.isfinite(values)).sum()),
    }

print("1. Numeric validity")
display(pd.DataFrame(numeric_checks).T)

# 2. Verify LineValue = Quantity × Price
expected_line_value = analysis["Quantity"] * analysis["Price"]
line_value_matches = np.isclose(
    analysis["LineValue"],
    expected_line_value,
    rtol=1e-10,
    atol=1e-8
)

print("\n2. Line-value reconciliation")
print("Matching rows:", f"{int(line_value_matches.sum()):,}")
print("Mismatching rows:", f"{int((~line_value_matches).sum()):,}")

# 3. Check whether numeric flags match their underlying values
flag_checks = {
    "IsNegativeQuantity": analysis["IsNegativeQuantity"].eq(
        analysis["Quantity"] < 0
    ),
    "IsZeroPrice": analysis["IsZeroPrice"].eq(
        analysis["Price"] == 0
    ),
    "IsNegativePrice": analysis["IsNegativePrice"].eq(
        analysis["Price"] < 0
    ),
    "MissingCustomerID": analysis["MissingCustomerID"].eq(
        analysis["Customer ID"].isna()
    ),
    "MissingDescription": analysis["MissingDescription"].eq(
        analysis["Description"].isna()
    ),
    "IsCancellationInvoice": analysis["IsCancellationInvoice"].eq(
        analysis["Invoice"].astype(str).str.strip().str.startswith("C")
    ),
}

print("\n3. Flag consistency")
for name, result in flag_checks.items():
    print(f"{name}: {bool(result.all())} | mismatches: {(~result).sum():,}")

# 4. Check dates and customer ID values
print("\n4. Date and customer ID checks")
print("Missing dates:", f"{analysis['InvoiceDate'].isna().sum():,}")
print("Missing customer IDs:", f"{analysis['Customer ID'].isna().sum():,}")

known_ids = analysis.loc[
    analysis["Customer ID"].notna(), "Customer ID"
]
fractional_ids = ~np.isclose(
    known_ids.to_numpy(dtype=float),
    np.round(known_ids.to_numpy(dtype=float))
)

print("Non-integer customer IDs:", f"{fractional_ids.sum():,}")

# 5. Check classification totals
print("\n5. Transaction classification")
display(
    analysis["TransactionType"]
    .value_counts(dropna=False)
    .rename_axis("TransactionType")
    .reset_index(name="row_count")
)

print("\nTotal classified rows:", f"{analysis['TransactionType'].notna().sum():,}")
print("Expected total rows:", f"{len(analysis):,}")


1. Numeric validity


,non_numeric,non_finite
Quantity,0,0
Price,0,0
LineValue,0,0



2. Line-value reconciliation
Matching rows: 1,067,371
Mismatching rows: 0

3. Flag consistency
IsNegativeQuantity: True | mismatches: 0
IsZeroPrice: True | mismatches: 0
IsNegativePrice: True | mismatches: 0
MissingCustomerID: True | mismatches: 0
MissingDescription: True | mismatches: 0
IsCancellationInvoice: True | mismatches: 0

4. Date and customer ID checks
Missing dates: 0
Missing customer IDs: 243,007
Non-integer customer IDs: 0

5. Transaction classification


,TransactionType,row_count
0,Merchandise candidate,1040749
1,Cancellation invoice,18850
2,Negative quantity / review,3457
3,Non-positive price / review,2738
4,Special / non-merchandise candidate,1577



Total classified rows: 1,067,371
Expected total rows: 1,067,371


In [73]:

# Final Data Quality Validation — Step 5
# Compare exported CSV files with their source DataFrames.

from pathlib import Path
import pandas as pd
import numpy as np

processed_dir = Path(
    r"D:\ai_powered_data_analyst\Retail_Analytics\data\processed"
)

# Reconstruct the expected source datasets
merchandise_primary = retail.loc[
    retail["TransactionType"] == "Merchandise candidate"
].copy()

merchandise_sensitivity = merchandise_primary.loc[
    ~merchandise_primary["IsExactDuplicateCandidate"]
].copy()

expected_exports = {
    "retail_classified_full.csv": retail,
    "retail_merchandise_primary.csv": merchandise_primary,
    "retail_merchandise_sensitivity.csv": merchandise_sensitivity,
    "retail_customer_analysis_candidates.csv": customer_analysis,
}

results = []

for filename, source_df in expected_exports.items():
    file_path = processed_dir / filename

    if not file_path.exists():
        print(f"\nMISSING FILE: {file_path}")
        continue

    exported_df = pd.read_csv(file_path)

    source_total = source_df["LineValue"].sum()
    exported_total = pd.to_numeric(
        exported_df["LineValue"], errors="coerce"
    ).sum()

    results.append({
        "file": filename,
        "source_rows": len(source_df),
        "exported_rows": len(exported_df),
        "row_count_match": len(source_df) == len(exported_df),
        "columns_match": list(source_df.columns) == list(exported_df.columns),
        "source_line_value": source_total,
        "exported_line_value": exported_total,
        "line_value_match": np.isclose(
            source_total, exported_total, rtol=1e-10, atol=0.01
        ),
    })

validation_results = pd.DataFrame(results)
display(validation_results)

if not validation_results.empty:
    check_columns = [
        "row_count_match",
        "columns_match",
        "line_value_match"
    ]

    print(
        "\nAll export checks passed:",
        bool(validation_results[check_columns].all().all())
        and len(validation_results) == len(expected_exports)
    )


,file,source_rows,exported_rows,row_count_match,columns_match,source_line_value,exported_line_value,line_value_match
0,retail_classified_full.csv,1067371,1067371,True,False,1.928725e+07,1.928725e+07,True
1,retail_merchandise_primary.csv,1040749,1040749,True,False,2.058626e+07,2.058626e+07,True
2,retail_merchandise_sensitivity.csv,1007024,1007024,True,False,2.010497e+07,2.010497e+07,True
3,retail_customer_analysis_candidates.csv,804808,804808,True,True,1.759062e+07,1.759062e+07,True



All export checks passed: False


In [75]:

# Find exactly which columns differ between source and exported CSVs

for filename, source_df in expected_exports.items():
    file_path = processed_dir / filename

    if not file_path.exists():
        continue

    exported_columns = pd.read_csv(file_path, nrows=0).columns.tolist()
    source_columns = source_df.columns.tolist()

    print(f"\n{'=' * 65}")
    print(filename)
    print("Source column count:", len(source_columns))
    print("Exported column count:", len(exported_columns))
    print("Same column names and order:", source_columns == exported_columns)

    missing_from_export = [
        col for col in source_columns if col not in exported_columns
    ]
    extra_in_export = [
        col for col in exported_columns if col not in source_columns
    ]

    print("Missing from export:", missing_from_export)
    print("Extra in export:", extra_in_export)

    if source_columns != exported_columns:
        print("\nFirst 10 source columns:", source_columns[:10])
        print("First 10 exported columns:", exported_columns[:10])



retail_classified_full.csv
Source column count: 19
Exported column count: 23
Same column names and order: False
Missing from export: []
Extra in export: ['Include_Merchandise_Analysis', 'Include_Customer_Analysis', 'Requires_Review', 'Duplicate_Sensitivity_Flag']

First 10 source columns: ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country', 'Description_clean', 'LineValue']
First 10 exported columns: ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country', 'Description_clean', 'LineValue']

retail_merchandise_primary.csv
Source column count: 19
Exported column count: 23
Same column names and order: False
Missing from export: []
Extra in export: ['Include_Merchandise_Analysis', 'Include_Customer_Analysis', 'Requires_Review', 'Duplicate_Sensitivity_Flag']

First 10 source columns: ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country', 'Description_

In [77]:

# Correct source-to-export mapping
# No files are modified.

expected_exports_corrected = {
    "retail_classified_full.csv": analysis,
    "retail_merchandise_primary.csv": analysis.loc[
        analysis["TransactionType"] == "Merchandise candidate"
    ],
    "retail_merchandise_sensitivity.csv": analysis.loc[
        (analysis["TransactionType"] == "Merchandise candidate")
        & (~analysis["IsExactDuplicateCandidate"])
    ],
    "retail_customer_analysis_candidates.csv": customer_analysis,
}

results_corrected = []

for filename, source_df in expected_exports_corrected.items():
    file_path = processed_dir / filename
    exported_df = pd.read_csv(file_path)

    source_total = source_df["LineValue"].sum()
    exported_total = exported_df["LineValue"].sum()

    results_corrected.append({
        "file": filename,
        "source_rows": len(source_df),
        "exported_rows": len(exported_df),
        "row_count_match": len(source_df) == len(exported_df),
        "columns_match": (
            list(source_df.columns) == list(exported_df.columns)
        ),
        "line_value_match": np.isclose(
            source_total, exported_total, rtol=1e-10, atol=0.01
        ),
    })

corrected_results = pd.DataFrame(results_corrected)
display(corrected_results)

checks = [
    "row_count_match",
    "columns_match",
    "line_value_match"
]

print(
    "\nAll four exports validated:",
    len(corrected_results) == 4
    and bool(corrected_results[checks].all().all())
)


,file,source_rows,exported_rows,row_count_match,columns_match,line_value_match
0,retail_classified_full.csv,1067371,1067371,True,True,True
1,retail_merchandise_primary.csv,1040749,1040749,True,True,True
2,retail_merchandise_sensitivity.csv,1007024,1007024,True,True,True
3,retail_customer_analysis_candidates.csv,804808,804808,True,True,True



All four exports validated: True
